# Lesson 3.3 — Path-Specific Rules for Conditional Convention Loading

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Create .claude/rules/ files with YAML frontmatter paths glob patterns
- Explain how path-scoped rules load only when editing matching files
- Prefer glob-pattern rules over directory CLAUDE.md for cross-directory conventions

**You will produce:** Two path-scoped rule files (API and test files) with glob frontmatter plus a matcher demo.

## Concepts

`.claude/rules/` files take a **YAML frontmatter `paths`** field with glob patterns for **conditional activation** - the rule loads only when you edit a matching file, reducing irrelevant context and token usage.

Sample Question 6: test files are spread throughout the codebase (e.g. `Button.test.tsx` next to `Button.tsx`). A rule with `paths: ["**/*.test.tsx"]` applies test conventions **by file type regardless of directory** - something directory-level CLAUDE.md files cannot do cleanly. Choose path-specific rules over subdirectory CLAUDE.md when conventions span many directories.

> **Reference:** [Anthropic - Claude Code rules/memory](https://docs.anthropic.com/en/docs/claude-code/memory)

In [ ]:
import os, pathlib
ROOT = pathlib.Path('cc_project'); ROOT.mkdir(exist_ok=True)
os.chdir(ROOT)
(pathlib.Path('.claude') / 'rules').mkdir(parents=True, exist_ok=True)
(pathlib.Path('.claude') / 'commands').mkdir(parents=True, exist_ok=True)
(pathlib.Path('.claude') / 'skills').mkdir(parents=True, exist_ok=True)
print('project root:', os.getcwd())

## Code-Along: Step 1 — Write path-scoped rule files

Each rule declares a glob in frontmatter; it loads only for matching files.

In [ ]:
api_rule = '''---
paths: ["src/api/**/*"]
---
# API Conventions
- async/await with explicit error handling
- Validate inputs at the boundary
'''
test_rule = '''---
paths: ["**/*.test.tsx", "**/*.test.ts"]
---
# Test Conventions
- Arrange-Act-Assert
- One behavior per test; descriptive names
'''
(pathlib.Path('.claude/rules') / 'api.md').write_text(api_rule, encoding='utf-8')
(pathlib.Path('.claude/rules') / 'tests.md').write_text(test_rule, encoding='utf-8')
print('wrote api.md and tests.md')

## Code-Along: Step 2 — Simulate conditional activation

We parse the globs and show which rules fire for a given edited file.

In [ ]:
import re, fnmatch  # parse frontmatter globs manually (no yaml dep)

def parse_paths(rule_text):
    m = re.search(r'paths:\s*\[(.*?)\]', rule_text)
    if not m:
        return []
    return [s.strip().strip('\"\'') for s in m.group(1).split(',')]

def active_rules(edited_file):
    active = []
    for rf in pathlib.Path('.claude/rules').glob('*.md'):
        globs = parse_paths(rf.read_text(encoding='utf-8'))
        if any(fnmatch.fnmatch(edited_file, g) for g in globs):
            active.append(rf.name)
    return active

print('editing src/api/orders.ts ->', active_rules('src/api/orders.ts'))
print('editing ui/Button.test.tsx ->', active_rules('ui/Button.test.tsx'))

## Code-Along: Step 3 — Rules vs directory CLAUDE.md

When files are scattered across directories, a glob rule beats directory-bound config.

In [ ]:
def choose(convention):
    return 'path-rule(glob)' if convention['spans_many_dirs'] else 'directory CLAUDE.md'

print('tests everywhere:', choose({'spans_many_dirs': True}))
print('one package only:', choose({'spans_many_dirs': False}))

## Exercise

Add a `terraform` rule with `paths: ["terraform/**/*"]`. Confirm it activates for `terraform/main.tf` but NOT for `src/api/orders.ts`. Then add an assertion that the test rule fires for a `.test.tsx` file located in two different directories.

In [ ]:
# TODO: write .claude/rules/terraform.md with paths: ["terraform/**/*"]
# TODO: assert 'terraform.md' in active_rules('terraform/main.tf')
# TODO: assert 'tests.md' in active_rules('a/b/Widget.test.tsx')

## Key Takeaways

- .claude/rules/ frontmatter `paths` globs load conventions only for matching files.
- Path-scoped rules cut irrelevant context and token usage.
- Glob rules apply by file type across directories - directory CLAUDE.md cannot.
- Prefer path-specific rules when conventions span many directories (e.g. test files).

---

**Next:** [Lesson 3.4 — Plan Mode vs Direct Execution](./lesson-3.4-plan-mode-vs-direct.ipynb)